# Knowledge Distillation (FSP + KD): ConvNeXt V2 → MobileNetV3 (Colab)

**Phương pháp:** 2 giai đoạn:
- **Stage 1 — FSP Pre-training** (Yim et al., CVPR 2017): học luồng biến đổi đặc trưng giữa teacher và student.
- **Stage 2 — KD Fine-tuning** (với ProjectionHead): cosine KD loss + MagFace task loss.

| | Teacher | Student |
|---|---|---|
| Model | `MTLFaceRecognition` (ConvNeXt V2) | `FaceRecognitionMobileNetV3` |
| Params | ~28M | ~5M |
| Embedding | `x_id` (512-D) | 512-D |
| Mode | **Frozen** | **Trainable** |

**Stage 1 — FSP matrix:**
$$G_{i,j}(x;W) = \frac{1}{H\times W}\sum_{s,t} F^1_{s,t,i}\times F^2_{s,t,j}$$
$$L_{FSP} = \frac{1}{N}\sum_i \|G_T^i - G_S^i\|_F^2$$

**Stage 2 — KD loss:**
$$L_{total} = \alpha\cdot L_{MagFace} + \beta\cdot L_{KD\_cos}$$

**2 giai đoạn training:**
```
Stage 1 — FSP Pre-training:
    Train: student backbone + adapter 1x1 conv (bỏ sau Stage 1)
    Không dùng MagFace, không cần id_labels

Stage 2 — KD Fine-tuning:
    L_total = alpha * L_MagFace + beta * L_KD_cosine
    ProjectionHead bridge student <-> teacher embedding space
    Load weights từ Stage 1, fine-tune toàn bộ student
```

**FSP pairs (raw timm backbone features):**
```
Teacher ConvNeXt V2 Tiny:  [96@28, 192@14, 384@7, 768@7]
  Pair 1: (feats[1]=192ch@14, feats[2]=384ch@7)
  Pair 2: (feats[2]=384ch@7,  feats[3]=768ch@7)

Student MobileNetV3 Large: [16@56, 24@28, 40@14, ?@7, 160@?]
  Pair 1: (feats[2]=40ch@14,  feats[3]=?ch@7)
  Pair 2: (feats[3]=?ch@7,    feats[4]=160ch@?)
```

**Cách dùng:**
1. Mount Google Drive (cell 1)
2. Sửa `CONFIGURATION` và `TEACHER_CKPT` (cell Config)
3. Chạy từ trên xuống — probe cell tự động xác định channel sizes

## 1. Mount Drive & Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.loss.FSPLoss import FSPDistillLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc,
    compute_id_auc_gallery_probe,
    compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

TEACHER_CKPT = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'

EXPERIMENT_NAME = 'KD_FSP_KD_ConvNextV2_to_MobileNetV3_Albedo'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,
    'output_dir':  '/content/drive/MyDrive/',

    # Modality: 'albedo' | 'normalmap' | 'depthmap'
    'type':        'albedo',

    'teacher_backbone': 'convnextv2_tiny',
    'backbone':         'mobilenetv3_large_100',

    'use_sampler': True,
    'device':      device,
    'image_size':  112,
    'num_classes': None,

    # ── Stage 1: FSP pre-training ──────────────────────────────────────
    'stage1_epochs':     40,
    'stage1_lr':         1e-4,
    'stage1_batch_size': 32,

    # ── Stage 2: KD fine-tuning với ProjectionHead ─────────────────────
    'stage2_epochs':     100,
    'stage2_lr':         1e-4,
    'stage2_batch_size': 32,
    'task_weight':       1.0,
    'kd_weight':         15.0,
}

print(f"Teacher ckpt : {TEACHER_CKPT}")
print(f"Stage 1      : {CONFIGURATION['stage1_epochs']} epochs, lr={CONFIGURATION['stage1_lr']}, batch={CONFIGURATION['stage1_batch_size']}")
print(f"Stage 2      : {CONFIGURATION['stage2_epochs']} epochs, lr={CONFIGURATION['stage2_lr']}, batch={CONFIGURATION['stage2_batch_size']}")

## 3. Data Loading

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'train_set.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(f'Không tìm thấy CSV train tại {dataset_dir}.')
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])
test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

CONFIGURATION['batch_size'] = CONFIGURATION['stage1_batch_size']
train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

## 4. Teacher Model (ConvNeXt V2 — Frozen)

In [ ]:
if not os.path.exists(TEACHER_CKPT):
    raise FileNotFoundError(f'Không tìm thấy teacher checkpoint: {TEACHER_CKPT}')

teacher = MTLFaceRecognition(
    backbone=CONFIGURATION['teacher_backbone'],
    num_classes=CONFIGURATION['num_classes'],
)

ckpt = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
state_dict = ckpt['model_state_dict']

keys_to_remove = [k for k in state_dict.keys() if 'id_head.maglinear' in k]
for k in keys_to_remove:
    del state_dict[k]

teacher.load_state_dict(state_dict, strict=False)
print(f"Teacher loaded — epoch {ckpt.get('epoch', '?')}")

teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'Teacher params: {teacher_params:,} (tất cả frozen)')

with torch.no_grad():
    _dummy = torch.randn(2, 3, 112, 112).to(device)
    _t_emb = teacher.get_embedding(_dummy)[-1]
    print(f'Teacher ID embedding shape: {_t_emb.shape}')

## 5. Probe Backbone Feature Shapes

Xác định channel/spatial sizes thực tế để cấu hình `FSPDistillLoss`.

In [ ]:
_student_probe = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
).to(device)

dummy = torch.randn(1, 3, 112, 112).to(device)

with torch.no_grad():
    t_feats = teacher.backbone.backbone(dummy)
    s_feats = _student_probe.backbone.backbone(dummy)

print('Teacher (ConvNeXt V2) timm backbone features:')
for i, f in enumerate(t_feats):
    print(f'  [{i}]: {tuple(f.shape)}  C={f.shape[1]}, spatial={f.shape[2]}x{f.shape[3]}')

print('\nStudent (MobileNetV3) timm backbone features:')
for i, f in enumerate(s_feats):
    print(f'  [{i}]: {tuple(f.shape)}  C={f.shape[1]}, spatial={f.shape[2]}x{f.shape[3]}')

del _student_probe

T_CH = [
    (t_feats[1].shape[1], t_feats[2].shape[1]),
    (t_feats[2].shape[1], t_feats[3].shape[1]),
]
S_CH = [
    (s_feats[2].shape[1], s_feats[3].shape[1]),
    (s_feats[3].shape[1], s_feats[4].shape[1]),
]

print(f'\nFSP pairs teacher channels : {T_CH}')
print(f'FSP pairs student channels : {S_CH}')

print('\nAdapter layers cần tạo:')
for k, ((ct1, ct2), (cs1, cs2)) in enumerate(zip(T_CH, S_CH)):
    print(f'  Pair {k+1}: f1 {cs1}→{ct1},  f2 {cs2}→{ct2}')

## 6. Student Model (MobileNetV3)

In [ ]:
student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
student.to(device)

total_p     = sum(p.numel() for p in student.parameters())
trainable_p = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student total params    : {total_p:,}')
print(f'Student trainable params: {trainable_p:,}')

## 7. Stage 1 — FSP Pre-training

$$L_{FSP} = \frac{1}{N}\sum_{i=1}^{N} \|G_T^i - G_S^i\|_F^2$$

- `FSPDistillLoss` chứa adapter 1×1 conv để project student channels → teacher dims
- Sau Stage 1: **chỉ lưu student weights** (adapter bỏ đi)

In [ ]:
fsp_criterion = FSPDistillLoss(
    teacher_ch_pairs=T_CH,
    student_ch_pairs=S_CH,
).to(device)

adapter_params = sum(p.numel() for p in fsp_criterion.parameters())
print(f'FSP adapter params: {adapter_params:,} (auxiliary — bỏ sau Stage 1)')

stage1_params = list(student.backbone.parameters()) + list(fsp_criterion.parameters())
optimizer_s1  = Adam(stage1_params, lr=CONFIGURATION['stage1_lr'])

CONFIGURATION['note'] = EXPERIMENT_NAME + '_Stage1'
manager_s1 = ExperimentManager(CONFIGURATION)
writer_s1  = SummaryWriter(log_dir=manager_s1.log_dir)
print(f'Stage 1 checkpoint dir: {manager_s1.ckpt_dir}')

In [ ]:
def train_stage1_epoch(train_dl, teacher, student, fsp_criterion, optimizer, device):
    student.train()
    fsp_criterion.train()
    total_loss = 0.0

    for X, _ in train_dl:
        X = X.to(device)

        with torch.no_grad():
            t_feats = teacher.backbone.backbone(X)

        s_feats = student.backbone.backbone(X)

        teacher_pairs = [
            (t_feats[1], t_feats[2]),
            (t_feats[2], t_feats[3]),
        ]
        student_pairs = [
            (s_feats[2], s_feats[3]),
            (s_feats[3], s_feats[4]),
        ]

        loss = fsp_criterion(teacher_pairs, student_pairs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_dl)

In [ ]:
manager_s1.log_text('BAT DAU STAGE 1 — FSP PRE-TRAINING')

for epoch in range(CONFIGURATION['stage1_epochs']):
    fsp_loss_val = train_stage1_epoch(
        train_dl, teacher, student, fsp_criterion, optimizer_s1, device
    )
    writer_s1.add_scalar('Loss/fsp', fsp_loss_val, epoch + 1)

    msg = f'[Stage1] Ep {epoch+1}/{CONFIGURATION["stage1_epochs"]}  fsp_loss={fsp_loss_val:.6f}'
    print(msg)
    manager_s1.log_text(msg)

writer_s1.close()

stage1_ckpt_path = os.path.join(manager_s1.ckpt_dir, 'stage1_student.pth')
torch.save(student.state_dict(), stage1_ckpt_path)
print(f'\nStage 1 xong. Student weights saved: {stage1_ckpt_path}')
manager_s1.log_text('STAGE 1 HOAN TAT.')

### Resume Stage 1 (nếu Colab disconnect)

> Chạy cell này để load lại stage1_student.pth, sau đó tiếp tục cell fit.

In [ ]:
# stage1_ckpt_path = '/content/drive/MyDrive/.../stage1_student.pth'

if os.path.exists(stage1_ckpt_path):
    student.load_state_dict(torch.load(stage1_ckpt_path, map_location=device))
    print(f'Loaded Stage 1 student: {stage1_ckpt_path}')
else:
    print(f'Không tìm thấy: {stage1_ckpt_path}')

## 8. Stage 2 — KD Fine-tuning

$$L_{total} = \alpha \cdot L_{MagFace} + \beta \cdot L_{KD\_cos}$$

Student đã được pre-train bởi FSP → weights là điểm khởi đầu tốt hơn random init.

| Loss | Ý nghĩa |
|---|---|
| `L_MagFace` | Class boundary với magnitude-aware margin |
| `L_KD_cos` | Point-wise: cosine distance giữa student (qua projector) và teacher embedding |

In [ ]:
class KDLoss(nn.Module):
    def __init__(self, metadata_path, task_weight=1.0, kd_weight=1.0):
        super().__init__()
        self.magface = WeightClassMagLoss(metadata_path)
        self.task_w  = task_weight
        self.kd_w    = kd_weight

    def forward(self, student_logits, student_norm, student_emb, teacher_emb, id_labels):
        l_task = self.magface(student_logits, id_labels, student_norm)
        s_n    = F.normalize(student_emb, p=2, dim=1)
        t_n    = F.normalize(teacher_emb, p=2, dim=1)
        l_kd   = (1.0 - F.cosine_similarity(s_n, t_n, dim=1)).mean()
        total  = self.task_w * l_task + self.kd_w * l_kd
        return total, l_task, l_kd


kd_criterion = KDLoss(
    metadata_path=train_csv,
    task_weight=CONFIGURATION['task_weight'],
    kd_weight=CONFIGURATION['kd_weight'],
)
print('KDLoss khởi tạo thành công.')

### ProjectionHead

Student và teacher sống trong hai embedding space khác nhau.

```
student_emb (512-D)  →  projector  →  proj_emb (512-D)  <->  teacher_emb
```

- `student_emb` vẫn đi thẳng vào `MagLinear` (task loss không thay đổi)
- Chỉ `proj_emb` được dùng cho KD loss
- Projector bỏ hoàn toàn khi export ONNX

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=512, out_dim=512):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.proj(x)


projector = ProjectionHead(in_dim=512, hidden_dim=512, out_dim=512).to(device)
proj_params = sum(p.numel() for p in projector.parameters())
print(f'ProjectionHead params: {proj_params:,}  (train-only, dropped at export)')

In [ ]:
CONFIGURATION['batch_size'] = CONFIGURATION['stage2_batch_size']
train_dl_s2, test_dl_s2, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Stage 2 — Train batches: {len(train_dl_s2)} | Test batches: {len(test_dl_s2)}')

optimizer_s2 = Adam(
    list(student.parameters()) + list(projector.parameters()),
    lr=CONFIGURATION['stage2_lr'],
)
scheduler_s2 = CosineAnnealingWarmRestarts(optimizer_s2, T_0=20, T_mult=2, eta_min=1e-6)

CONFIGURATION['note'] = EXPERIMENT_NAME + '_Stage2'
manager_s2 = ExperimentManager(CONFIGURATION)
writer_s2  = SummaryWriter(log_dir=manager_s2.log_dir)

ckpt_saver_s2 = ModelCheckpoint(
    output_dir=manager_s2.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping_s2 = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=10,
    mode='max',
    verbose=1,
    save_dir=manager_s2.ckpt_dir,
    start_from_epoch=5,
)
print(f'Stage 2 checkpoint dir: {manager_s2.ckpt_dir}')

In [ ]:
def train_stage2_epoch(train_dl, teacher, student, projector, criterion, optimizer, device):
    student.train()
    projector.train()
    total_loss = total_task = total_kd = 0.0

    for X, y in train_dl:
        X, y      = X.to(device), y.to(device)
        id_labels = y[:, 0]

        with torch.no_grad():
            teacher_emb = teacher.get_embedding(X)[-1]

        feat         = student.backbone(X)
        student_emb  = student.embedding(feat)
        proj_emb     = projector(student_emb)
        logits, norm = student.maglinear(student_emb)

        loss, l_task, l_kd = criterion(logits, norm, proj_emb, teacher_emb, id_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_task += l_task.item()
        total_kd   += l_kd.item()

    n = len(train_dl)
    return total_loss/n, total_task/n, total_kd/n


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [ ]:
START_EPOCH_S2 = 0
manager_s2.log_text('BAT DAU STAGE 2 — KD FINE-TUNING (init tu FSP Stage 1)')

for epoch in range(START_EPOCH_S2, CONFIGURATION['stage2_epochs']):
    manager_s2.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["stage2_epochs"]} ---')

    train_loss, train_task, train_kd = train_stage2_epoch(
        train_dl_s2, teacher, student, projector, kd_criterion, optimizer_s2, device
    )

    train_auc = compute_id_auc(train_dl_s2, student, device)
    test_auc  = compute_id_auc(test_dl_s2,  student, device)

    train_metrics = {
        'loss':             train_loss,
        'loss_task':        train_task,
        'loss_kd':          train_kd,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
        'auc_id_euclidean': test_auc['id_euclidean'],
    }

    writer_s2.add_scalar('Loss/total', train_loss, epoch + 1)
    writer_s2.add_scalar('Loss/task',  train_task, epoch + 1)
    writer_s2.add_scalar('Loss/kd',    train_kd,   epoch + 1)
    writer_s2.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer_s2.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager_s2.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    ckpt_saver_s2(
        student, optimizer_s2, epoch + 1, test_metrics, scheduler_s2,
        extra_state={'projector_state_dict': projector.state_dict()},
    )
    early_stopping_s2(test_metrics, student, epoch + 1)
    scheduler_s2.step(epoch)

    if early_stopping_s2.early_stop:
        manager_s2.log_text('Early stopping triggered.')
        break

writer_s2.close()
manager_s2.log_text('STAGE 2 HOAN TAT.')

### Resume Stage 2 (nếu Colab disconnect)

> Chạy Setup → Imports → Data → Teacher → Probe → Student → Stage 2 Setup,
> sau đó chạy cell này, rồi chạy lại cell fit.

In [ ]:
CKPT_PATH = os.path.join(manager_s2.ckpt_dir, 'last_model.pth')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

checkpoint = torch.load(CKPT_PATH, map_location=device)
student.load_state_dict(checkpoint['model_state_dict'])
optimizer_s2.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler_s2.load_state_dict(checkpoint['scheduler_state_dict'])
if 'projector_state_dict' in checkpoint:
    projector.load_state_dict(checkpoint['projector_state_dict'])
    print('Projector state loaded.')

START_EPOCH_S2 = checkpoint['epoch']
print(f'Resume Stage 2 tu epoch {START_EPOCH_S2} — chay lai cell fit de tiep tuc.')

## 9. Đánh giá Final (Gallery-Probe)

In [ ]:
gallery_dl, probe_dl = create_eval_loaders(CONFIGURATION, test_transform)
print(f'Gallery batches: {len(gallery_dl)} | Probe batches: {len(probe_dl)}')

In [ ]:
class _TeacherWrapper(torch.nn.Module):
    def __init__(self, t): super().__init__(); self._t = t
    def get_embedding(self, x): return self._t.get_embedding(x)[-1]

teacher_wrapper = _TeacherWrapper(teacher).to(device)
teacher_wrapper.eval()

teacher_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)
teacher_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)

In [ ]:
best_ckpt_path = os.path.join(manager_s2.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()
print(f"Best model tu epoch {best_ckpt.get('epoch', '?')}")

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

print(tabulate([
    ['Model',          CONFIGURATION['teacher_backbone'],        CONFIGURATION['backbone']],
    ['Params',         f'{teacher_params:,}',                    f'{total_p:,}'],
    ['Cosine AUC',     f"{teacher_gp_auc['id_cosine']:.4f}",    f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC',  f"{teacher_gp_auc['id_euclidean']:.4f}", f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc',     f'{teacher_gp_rank1:.4f}',               f'{student_gp_rank1:.4f}'],
], headers=['Metric', 'Teacher', 'Student (FSP+KD)'], tablefmt='fancy_grid'))

## 10. Export ONNX

In [ ]:
class InferenceWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.backbone  = model.backbone
        self.embedding = model.embedding

    def forward(self, x):
        return F.normalize(self.embedding(self.backbone(x)), p=2, dim=1)


inference_model = InferenceWrapper(student).eval().cpu()
dummy_input = torch.randn(1, 3, 112, 112)

onnx_path = os.path.join(manager_s2.ckpt_dir, 'fsp_kd_mobilenetv3_fr.onnx')
torch.onnx.export(
    inference_model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {onnx_path}')